In [22]:
import pandas as pd
import sys
sys.path.append("..")
from src.text_cleaning import clean_text

df = pd.read_csv('../data/eval/Resume.csv')
df

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR
...,...,...,...,...
2479,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2480,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION
2481,31605080,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2482,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION


In [23]:
df['clean_text'] = df['Resume_str'].apply(clean_text)
print(df['clean_text'][0])

hr administrator marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq micros opera pms fidelio opera reservation system ors holidex completed courses and seminars in customer service sales strategies inventory control loss prevention safety time management leadership and performance assessment experience hr administrator marketin

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [25]:
vectorizer = TfidfVectorizer(token_pattern=r'\S+')
X = vectorizer.fit_transform(df['clean_text'])
X.shape

(2484, 43475)

In [26]:
from sklearn.model_selection import train_test_split

y = df['Category']

texts_train, texts_test, y_train, y_test = train_test_split(
    df['clean_text'],y, test_size=0.2, random_state=42, stratify=y
)

print(texts_train.shape)
print(texts_test.shape)
print(y_train.shape)
print(y_test.shape)

(1987,)
(497,)
(1987,)
(497,)


In [27]:
clf_vectorizer = TfidfVectorizer(token_pattern=r'\S+')

X_train = clf_vectorizer.fit_transform(texts_train)
X_test = clf_vectorizer.transform(texts_test)
print(X_train.shape)
print(X_test.shape)

(1987, 38638)
(497, 38638)


In [28]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight= "balanced"
)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [29]:
y_pred = model.predict(X_test)

In [30]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.67      0.83      0.74        24
              ADVOCATE       0.43      0.42      0.43        24
           AGRICULTURE       0.57      0.62      0.59        13
               APPAREL       0.67      0.21      0.32        19
                  ARTS       0.50      0.29      0.36        21
            AUTOMOBILE       0.50      0.57      0.53         7
              AVIATION       0.85      0.71      0.77        24
               BANKING       0.88      0.65      0.75        23
                   BPO       0.67      0.50      0.57         4
  BUSINESS-DEVELOPMENT       0.49      0.79      0.60        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.83      0.91      0.87        22
            CONSULTANT       0.60      0.13      0.21        23
              DESIGNER       0.85      0.81      0.83        21
         DIGITAL-MEDIA       0.73      

In [31]:

from sklearn.metrics.pairwise import cosine_similarity

def search(vacancy_text, k=5):
    vacancy_clean = clean_text(vacancy_text)
    vacancy_vec = vectorizer.transform([vacancy_clean])
    scores = cosine_similarity(vacancy_vec, X)[0]
    result = df.copy()
    result['score'] = scores
    return result.sort_values('score', ascending=False).head(k)



In [32]:
vacancies = {
    "ACCOUNTANT": "Accountant to manage general ledger, accounts payable and receivable, tax returns, reconciliations and monthly financial reporting.",
    "ADVOCATE": "Attorney to provide legal advice, draft contracts, conduct legal research and represent clients in court and litigation.",
    "AGRICULTURE": "Agriculture specialist to manage crop production, farm operations, soil and irrigation, livestock and agronomy programs.",
    "APPAREL": "Apparel merchandiser to manage clothing collections, garment production, fashion trends, fabric sourcing and retail buying.",
    "ARTS": "Arts program coordinator to organize exhibitions, theater and music events, work with artists and manage art education programs.",
    "AUTOMOBILE": "Automotive technician to diagnose and repair vehicles, engines, brakes and transmissions, and perform vehicle maintenance.",
    "AVIATION": "Aviation maintenance technician to inspect and repair aircraft, follow FAA regulations and support flight operations.",
    "BANKING": "Banking officer to open customer accounts, process loans and deposits, ensure compliance and provide branch customer service.",
    "BPO": "BPO call center agent to handle inbound and outbound customer calls, resolve client issues and meet process quality targets.",
    "BUSINESS-DEVELOPMENT": "Business development manager to find new clients, build partnerships, generate leads and grow company revenue.",
    "CHEF": "Chef to prepare dishes, plan menus, manage the kitchen staff, ensure food safety and control food costs.",
    "CONSTRUCTION": "Construction project manager to supervise building sites, contractors, safety, schedules and construction budgets.",
    "CONSULTANT": "Consultant to analyze client business problems, recommend solutions, lead projects and present findings to stakeholders.",
    "DESIGNER": "Designer to create visual concepts, layouts and branding using Adobe Photoshop, Illustrator and InDesign.",
    "DIGITAL-MEDIA": "Digital media specialist to manage social media, create video and web content, run online campaigns and analyze engagement.",
    "ENGINEERING": "Engineer to design mechanical and electrical systems, use AutoCAD, perform testing and manage engineering projects.",
    "FINANCE": "Financial analyst to build financial models, forecasts and budgets, analyze investments and prepare financial statements.",
    "FITNESS": "Fitness trainer to lead personal training sessions and group exercise classes, design workout programs and coach clients.",
    "HEALTHCARE": "Healthcare professional to provide patient care, maintain medical records, assist physicians and follow clinical procedures.",
    "HR": "HR specialist to manage recruitment, onboarding, employee relations, payroll, benefits and HR policies.",
    "INFORMATION-TECHNOLOGY": "IT specialist to support networks, servers and help desk, manage systems administration and troubleshoot hardware and software.",
    "PUBLIC-RELATIONS": "Public relations specialist to write press releases, manage media relations, plan events and handle corporate communications.",
    "SALES": "Sales representative to sell products, meet sales targets, manage customer accounts and negotiate deals.",
    "TEACHER": "Teacher to plan lessons, teach students in the classroom, assess student progress and communicate with parents.",
}

results = []

for category, text in vacancies.items():
    top = search(text)
    precision = (top['Category'] == category).mean()
    results.append({"category": category, "precision@5": precision})

results_df = pd.DataFrame(results)
results_df

,category,precision@5
0,ACCOUNTANT,1.0
1,ADVOCATE,0.0
2,AGRICULTURE,1.0
3,APPAREL,0.6
4,ARTS,1.0
5,AUTOMOBILE,0.2
6,AVIATION,0.8
7,BANKING,0.8
8,BPO,0.2
9,BUSINESS-DEVELOPMENT,1.0


In [33]:
search(vacancies['ADVOCATE'])[['Category', 'score']]

,Category,score
1333,AUTOMOBILE,0.382026
1126,CONSULTANT,0.327789
1662,APPAREL,0.253655
2449,AVIATION,0.250288
1342,AUTOMOBILE,0.229967
